1️⃣ Imports & Setup

In [1]:
# ==============================
# 1️⃣ Imports & Setup
# ==============================
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
import warnings
warnings.filterwarnings("ignore")  # hide warnings for clean output

# 2️⃣ Load Dataset
loading sales dataset and show a sample of the first 10 rows.

In [2]:
# Load CSV
df = pd.read_csv("sales_data.csv")
display(df.head(10))

,Date,Product,Region,Sales,Customer_Age,Customer_Gender,Customer_Satisfaction
0,2022-01-01,Widget C,South,786,26,Male,2.874407
1,2022-01-02,Widget D,East,850,29,Male,3.365205
2,2022-01-03,Widget A,North,871,40,Female,4.547364
3,2022-01-04,Widget C,South,464,31,Male,4.555420
4,2022-01-05,Widget C,South,262,50,Female,3.982935
5,2022-01-06,Widget D,East,147,35,Female,1.140628
6,2022-01-07,Widget A,North,610,66,Male,3.300282
7,2022-01-08,Widget A,South,428,58,Male,4.146334
8,2022-01-09,Widget C,West,939,26,Male,1.069152
9,2022-01-10,Widget B,West,215,40,Male,3.198738


# 3️⃣ Interactive Dashboard Setup
Creating a dashboard with interactive filters for Region and Product. 
Selecting a filter updates the KPIs and charts automatically.

In [3]:
# Unique filter options
regions = ["All"] + list(df["Region"].unique())
products = ["All"] + list(df["Product"].unique())

# Dropdown widgets
region_dropdown = widgets.Dropdown(options=regions, value="All", description="Region:")
product_dropdown = widgets.Dropdown(options=products, value="All", description="Product:")

display(region_dropdown, product_dropdown)

Dropdown(description='Region:', options=('All', 'South', 'East', 'North', 'West'), value='All')

Dropdown(description='Product:', options=('All', 'Widget C', 'Widget D', 'Widget A', 'Widget B'), value='All')

In [4]:
# Function to update dashboard based on filters
def update_dashboard(region, product):
    clear_output(wait=True)  # clear previous output
    display(region_dropdown, product_dropdown)  # keep dropdowns visible
    
    # Filter data
    filtered_df = df.copy()
    if region != "All":
        filtered_df = filtered_df[filtered_df["Region"] == region]
    if product != "All":
        filtered_df = filtered_df[filtered_df["Product"] == product]
    
    # ==============================
    # KPIs
    # ==============================
    total_sales = filtered_df["Sales"].sum()
    top_product = filtered_df.groupby("Product")["Sales"].sum().idxmax() if not filtered_df.empty else "N/A"
    top_region = filtered_df.groupby("Region")["Sales"].sum().idxmax() if not filtered_df.empty else "N/A"
    avg_satisfaction = filtered_df["Customer_Satisfaction"].mean().round(2) if not filtered_df.empty else 0
    avg_customer_age = filtered_df["Customer_Age"].mean().round(1) if not filtered_df.empty else 0
    
    display(Markdown("## 📊 Key Metrics"))
    display(Markdown(f"- **Total Sales:** ${total_sales:,.0f}"))
    display(Markdown(f"- **Top Product:** {top_product}"))
    display(Markdown(f"- **Top Region:** {top_region}"))
    display(Markdown(f"- **Average Customer Satisfaction:** {avg_satisfaction} / 5"))
    display(Markdown(f"- **Average Customer Age:** {avg_customer_age}"))
    
    # ==============================
    # Charts
    # ==============================
    # Sales by Product
    display(Markdown("### Sales by Product"))
    sales_by_product = filtered_df.groupby("Product")["Sales"].sum()
    fig, ax = plt.subplots()
    sales_by_product.plot(kind='bar', ax=ax, color='skyblue')
    ax.set_ylabel("Sales")
    ax.set_title("Total Sales by Product")
    plt.show()
    
    # Sales by Region
    display(Markdown("### Sales by Region"))
    sales_by_region = filtered_df.groupby("Region")["Sales"].sum()
    fig, ax = plt.subplots()
    sales_by_region.plot(kind='bar', ax=ax, color='lightgreen')
    ax.set_ylabel("Sales")
    ax.set_title("Total Sales by Region")
    plt.show()
    
    # Customer Satisfaction by Product
    display(Markdown("### Customer Satisfaction by Product"))
    satisfaction_by_product = filtered_df.groupby("Product")["Customer_Satisfaction"].mean()
    fig, ax = plt.subplots()
    satisfaction_by_product.plot(kind='bar', ax=ax, color='orange')
    ax.set_ylabel("Avg Satisfaction")
    ax.set_title("Customer Satisfaction by Product")
    plt.show()

In [5]:
# Connect dropdowns to update function
widgets.interactive(update_dashboard, region=region_dropdown, product=product_dropdown)

interactive(children=(Dropdown(description='Region:', options=('All', 'South', 'East', 'North', 'West'), value…

# 4️⃣ AI-Powered Q&A
Setting up an AI assistant using LangChain and OpenAI.  
It can answer questions about filtered or full dataset.

In [6]:
# Convert dataset rows to text for AI retriever
data_texts = df.apply(
    lambda row: f"Date: {row['Date']}, Product: {row['Product']}, Region: {row['Region']}, Sales: {row['Sales']}, Customer Age: {row['Customer_Age']}, Customer Gender: {row['Customer_Gender']}, Customer Satisfaction: {row['Customer_Satisfaction']}",
    axis=1
).tolist()

# Create vector store for RAG
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(data_texts, embeddings)

# Initialize AI LLM
llm = ChatOpenAI(temperature=0.7, model="gpt-3.5-turbo")
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())

In [ ]:
# Interactive AI Q&A
display(Markdown("## 💬 Ask AI about your Business Data"))

while True:
    question = input("Type your question (or 'exit' to quit): ")
    if question.lower() == "exit":
        print("Exiting AI assistant. ✅")
        break
    answer = qa.run(question)
    display(Markdown(f"**AI Answer:** {answer}"))

## 💬 Ask AI about your Business Data

Filter-aware AI Q&A

1️⃣ Function to create filtered retriever

In [ ]:
def get_filtered_qa(filtered_df):
    # Convert filtered rows to text
    data_texts = filtered_df.apply(
        lambda row: f"Date: {row['Date']}, Product: {row['Product']}, Region: {row['Region']}, Sales: {row['Sales']}, Customer Age: {row['Customer_Age']}, Customer Gender: {row['Customer_Gender']}, Customer Satisfaction: {row['Customer_Satisfaction']}",
        axis=1
    ).tolist()
    
    # Create vectorstore retriever
    vectorstore = FAISS.from_texts(data_texts, embeddings)
    
    # Create QA chain
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())
    return qa

2️⃣ Function to run interactive AI with dropdown filters

In [ ]:
def interactive_ai(region, product):
    # Filter data based on current selections
    filtered_df = df.copy()
    if region != "All":
        filtered_df = filtered_df[filtered_df["Region"] == region]
    if product != "All":
        filtered_df = filtered_df[filtered_df["Product"] == product]
    
    # Create filter-aware QA
    qa_filtered = get_filtered_qa(filtered_df)
    
    display(Markdown(f"### 💬 Ask AI about {region} / {product} data"))
    
    while True:
        question = input("Type your question (or 'exit' to quit): ")
        if question.lower() == "exit":
            print("Exiting AI assistant. ✅")
            break
        answer = qa_filtered.run(question)
        display(Markdown(f"**AI Answer:** {answer}"))

3️⃣ Connect AI to dashboard dropdowns

In [ ]:
# Example: run AI for current selections
widgets.interactive(interactive_ai, region=region_dropdown, product=product_dropdown)